In [41]:
#Phase 1
#Df compplet

from sklearn.datasets import load_breast_cancer
import numpy as np

def explorer_dataset():
    data = load_breast_cancer(return_X_y=False)

    X = data.data
    y = data.target
    names = data.target_names  # ['malin', 'benin']

    print("Lignes, colonnes :", X.shape)

    classes, counts = np.unique(y, return_counts=True) #résultat =class [0,1] count[212,357]
    total = len(y) # 569

    for classe, count in zip(classes, counts):
        pourcentage = count / total * 100
        print(f"Classe {classe} ({names[classe]}) : {count} cas ({pourcentage:.1f} %)")

explorer_dataset()

Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3 %)
Classe 1 (benign) : 357 cas (62.7 %)


In [42]:
#Dataset limité (filtrer sur y == 0)
def cas_limite():
    data = load_breast_cancer()
    X, y = data.data, data.target
    names = data.target_names

    X_f = X[y == 0]
    y_f = y[y == 0]

    print("Lignes, colonnes :", X_f.shape)
    print(f"Classe 0 ({names[0]}) : {len(y_f)} cas (100%)")

cas_limite()

Lignes, colonnes : (212, 30)
Classe 0 (malignant) : 212 cas (100%)


In [43]:
#CAS adversarial (datasert trompeur)
def cas_adversarial():
    y_fake = np.array([0]*540 + [1]*29)

    classes, counts = np.unique(y_fake, return_counts=True)
    total = len(y_fake)

    for c, n in zip(classes, counts):
        print(f"Classe {c} : {n} cas ({n/total*100:.1f} %)")

cas_adversarial()


#Conclusion : Le Dataset nest pas vraiment équilibré 212 bénin / 357 malin il est quand même nécessaire
# d'analyser les scores en % car ce déséquilibre fausse les résultats et performance du modèle ...

Classe 0 : 540 cas (94.9 %)
Classe 1 : 29 cas (5.1 %)


In [ ]:
# Phase 2 
#Nous allons entrainner le modèle a faire des prédiction et calculer un accuracy (Précision)

In [ ]:
from sklearn.metrics import accuracy_score

# train données auquelles a accés le model 
# test donnnées "réel" sur lesquelles on évalue le modèle
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    
    # 1. Entraînement
    modele.fit(X_train, y_train)

    # 2. Prédiction
    y_pred = modele.predict(X_test)

    # 3. Évaluation
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

# dataset
data = load_breast_cancer()
X, y = data.data, data.target

# split  test size 20% de donnée réel, 80% pour entrainement
# random state : mélange au hasard a partir de 42, donc toujours le même mélange donc toujours le meme résultat
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# modèle random forest
model = RandomForestClassifier()

# évaluation
score = entrainer_et_evaluer(model, X_train, X_test, y_train, y_test)

print("Accuracy :", score)

Accuracy : 0.9649122807017544


Résultat : Le modèle de classification prédit la classe dans 96,49% des cas les tumeur bénigne et maligne, autrement dit, il se trompe dans 3.50% des cas 

Phase 3

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return accuracy_score(y_test, y_pred)

def arene(X_train, X_test, y_train, y_test):

    modeles = {
        "Régression logistique": LogisticRegression(max_iter=5000), # max iter = 1000 = movaise iptimisation
        "KNN": KNeighborsClassifier(),
        "Arbre de décision": DecisionTreeClassifier()
    }

    resultats = []

    for nom, modele in modeles.items():
        score = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        resultats.append((nom, score))

    resultats.sort(key=lambda x: x[1], reverse=True)

    print("Classement des modèles :\n")

    for i, (nom, score) in enumerate(resultats, 1):
        print(f"{i}. {nom} : {score*100:.2f}%")

    return resultats


# DATASET
data = load_breast_cancer()
X, y = data.data, data.target

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# IMPORTANT : appel
arene(X_train, X_test, y_train, y_test)

🏆 Classement des modèles :

1. Régression logistique : 95.61%
2. KNN : 95.61%
3. Arbre de décision : 94.74%


[('Régression logistique', 0.956140350877193),
 ('KNN', 0.956140350877193),
 ('Arbre de décision', 0.9473684210526315)]

Les résultats sont plutôt satisfaisant ! Vous appercevrez un similarité entre les résultats de de la régression logistique et kNN, cela me semble très surprenant et je pense avoir fait une erreur. En effet la probabilité que deux modèles est la même précision est faible...


Phase 4:

In [59]:
from sklearn.cluster import KMeans

def clustering_aveugle(X):
    #regrouper les données en 2 clusters sans étiquettes !

    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)

    labels = kmeans.fit_predict(X)

    return labels


Le clustering retrouve paetiellement les vraie classes, ce qui démontre un structure dans les données. Cepndnant, comme l'algorythme n'est pas supervisé, il aussi être influancé par le "hasard"